In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# df = pd.read_csv("Data/split_datasets/cure_ckd_egfr_registry_preprocessed_project_preproc_data_UCLA_test.csv")
df = pd.read_csv("../Data/cure_ckd_egfr_registry_preprocessed_project_preproc_data_excl_crit_applied.csv")
df.head()

In [ ]:
cols = [col for col in df.columns if "_norace_reduction_40_ge" in col]
cols

In [ ]:
cols_egfr = [col for col in df.columns if "_norace_mean" in col]
cols_egfr

In [ ]:
df[cols].describe()

In [ ]:
(df[cols[:7]].sum(axis=1) > 5).sum()

In [ ]:
cond = (df[cols[:7]].sum(axis=1) < 2) & (df[cols[:7]].sum(axis=1) > 0)
df.loc[cond, cols_egfr[:8] + cols[:7]].head(n=5)

In [ ]:
egfr_cols = cols_egfr
outcome_cols = cols

# 1. Fluctuations: Count how many times eGFR goes UP year-over-year
# Assuming egfr_cols = ['eGFR_0', 'eGFR_1', ..., 'eGFR_6']
upswing_counts = []

for i in range(1, 7):
    current_year = egfr_cols[i]
    prev_year = egfr_cols[i-1]
    
    # Check where both years are present (not NaN) AND eGFR increased
    # This captures true recovery/fluctuation events
    upswing = ((df[current_year] > df[prev_year]) & 
               (df[current_year].notna()) & 
               (df[prev_year].notna())).sum()
    
    upswing_counts.append(upswing)
    print(f"Year {i-1} to Year {i} Upswings: {upswing}")

print(f"Total documented upswings across 6 years: {sum(upswing_counts)}")

In [ ]:
# Assuming egfr_cols = ['eGFR_0', 'eGFR_1', ..., 'eGFR_6']
significant_upswing_counts = []

for i in range(1, 7):
    current_year = egfr_cols[i]
    prev_year = egfr_cols[i-1]
    
    # Logic: 
    # 1. Both years must have data (not NaN)
    # 2. The increase must be > 20% of the previous year's value
    upswing_mask = (
        (df[current_year].notna()) & 
        (df[prev_year].notna()) & 
        (df[current_year] > (df[prev_year] * 1.20))
    )
    
    upswing_count = upswing_mask.sum()
    significant_upswing_counts.append(upswing_count)
    print(f"Year {i-1} to Year {i} Significant Upswings (>20%): {upswing_count}")

print(f"Total significant upswings (>20%) across 6 years: {sum(significant_upswing_counts)}")

In [ ]:
# Assuming egfr_cols = ['eGFR_0', ..., 'eGFR_6']
# Assuming outcome_cols = ['decline_40_1', ..., 'decline_40_6']

def analyze_recurrent_dynamics(df, num_decl = 1):
    # 1. Identify patients with MORE THAN ONE decline event
    # Sum across the 6-year outcome columns
    df['total_decline_events'] = df[outcome_cols].sum(axis=1)
    more_than_one_decline = df[df['total_decline_events'] > num_decl]
    
    # 2. Identify significant upswings (>20%) for these specific patients
    has_upswing = pd.Series(False, index=more_than_one_decline.index)
    for i in range(1, 7):
        current = more_than_one_decline[egfr_cols[i]]
        prev = more_than_one_decline[egfr_cols[i-1]]
        
        # Condition: Valid data in both years AND >20% increase
        condition = (current.notna()) & (prev.notna()) & (current > (prev * 1.20))
        has_upswing = has_upswing | condition
    
    # 3. Calculate intersection
    multi_decline_with_upswing = more_than_one_decline[has_upswing]
    
    # --- Statistics for Reviewer Response ---
    count_multi = len(more_than_one_decline)
    count_recovered = len(multi_decline_with_upswing)
    recovery_rate = (count_recovered / count_multi) * 100 if count_multi > 0 else 0
    
    print(f"Total patients with >{num_decl} decline event (Recurrence): {count_multi}")
    print(f"Of those, patients who also had a >20% upswing (Recovery): {count_recovered}")
    print(f"Recovery Rate among recurrent decliners: {recovery_rate:.2f}%")

for i in range(1,5):
    print(f"For more than {i}")
    analyze_recurrent_dynamics(df, num_decl=i)

In [ ]:
def analyze_baseline_recovery(df, num_decl=1):
    # 1a. Identify patients who ever hit the >=40% decline threshold
    df['ever_declined'] = df[outcome_cols].any(axis=1)
    decliners_df = df[df['ever_declined']].copy()

    # 1b. Identify patients with MORE THAN ONE decline event
    # Sum across the 6-year outcome columns
    df['total_decline_events'] = df[outcome_cols].sum(axis=1)
    more_than_one_decline = df[df['total_decline_events'] > num_decl]
    
    # 2. Define the recovery threshold: eGFR must be > 60% of eGFR_0
    # (If eGFR is > 60% of baseline, they are no longer in a >=40% reduction state)
    decliners_df['recovery_limit'] = decliners_df[egfr_cols[0]] * 0.60
    
    def check_recovery_path(row):
        has_hit_40 = False
        recovered_after_hit = False
        
        for i in range(1, 7):
            current_egfr = row[egfr_cols[i]]
            if pd.isna(current_egfr): continue
            
            # Did they hit the 40% decline threshold this year?
            if row[outcome_cols[i]] == 1:
                has_hit_40 = True
            
            # If they previously hit it, did they later recover above the 40% limit?
            elif has_hit_40 and (current_egfr > row['recovery_limit']):
                recovered_after_hit = True
                
        return recovered_after_hit

    decliners_df['has_recovered_from_40'] = decliners_df.apply(check_recovery_path, axis=1)
    
    # --- Statistics for Reviewer ---
    total_decliners = len(decliners_df)
    count_multi = len(more_than_one_decline)
    recovered_count = decliners_df['has_recovered_from_40'].sum()
    
    print(f"Total patients who experienced a >=40% decline: {total_decliners}")
    print(f"Total patients with >{num_decl} decline event (Recurrence): {count_multi}")
    print(f"Patients who later recovered (eGFR > 60% of baseline): {recovered_count}")
    print(f"Recovery Rate: {(recovered_count/total_decliners)*100:.2f}%")

for i in range(1,5):
    print(f"For more than {i}")
    analyze_baseline_recovery(df, num_decl=i)


In [ ]:
def analyze_recurrent_recovery(df, num_decl=1):
    # 1. Identify patients with MORE THAN ONE decline event (>1)
    df['total_decline_events'] = df[outcome_cols].sum(axis=1)
    recurrent_df = df[df['total_decline_events'] > num_decl].copy()
    
    # 2. Define the recovery threshold relative to initial baseline
    # (If eGFR is > 60% of Year 0, they are no longer in a >=40% reduction state)
    recurrent_df['recovery_limit'] = recurrent_df[egfr_cols[0]] * 0.60
    
    def check_recovery_trajectory(row):
        has_previously_hit_40 = False
        recovered_at_any_point = False
        
        # Walk through the 6-year trajectory
        for i in range(1, 7):
            current_egfr = row[egfr_cols[i]]
            current_outcome = row[outcome_cols[i-1]] # Adjusting index if outcome_cols starts at Yr1
            
            if pd.isna(current_egfr): continue
            
            # If they are currently in a 40% decline state
            if current_outcome == 1:
                has_previously_hit_40 = True
            
            # If they hit it before, check if this specific year shows recovery
            elif has_previously_hit_40 and (current_egfr > row['recovery_limit']):
                recovered_at_any_point = True
                
        return recovered_at_any_point

    recurrent_df['has_recovered'] = recurrent_df.apply(check_recovery_trajectory, axis=1)
    
    # --- Statistics for Reviewer Response ---
    count_multi = len(recurrent_df)
    count_recovered_multi = recurrent_df['has_recovered'].sum()
    recovery_rate = (count_recovered_multi / count_multi) * 100 if count_multi > 0 else 0
    
    print(f"Total patients with >{num_decl} decline event (Recurrence): {count_multi}")
    print(f"Recurrent patients who recovered to >60% of baseline at some point: {count_recovered_multi}")
    print(f"Recovery Rate among Recurrent Decliners: {recovery_rate:.2f}%")

for i in range(1,5):
    print(f"For more than {i}")
    analyze_recurrent_recovery(df, num_decl=i)
